## 1. Preparacion_Ambiente

Crea las external locations, el catalogo `football_dev` y las tablas Delta vacias
de las capas bronze, silver y gold para el pipeline de futbol Big Five.

Usa un catalogo distinto a `catalog_au` (proyecto F1) para no borrar ese ambiente.


In [0]:
dbutils.widgets.removeAll()


In [0]:
%sql
create widget text storageName default "adlssmartdata1702";


In [0]:
%python
storageName = dbutils.widgets.get("storageName")


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicacion externa para el metastore del Data Lake';


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicacion externa para los archivos raw del Data Lake';


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicacion externa para las tablas bronze del Data Lake';


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicacion externa para las tablas silver del Data Lake';


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicacion externa para las tablas golden del Data Lake';


In [0]:
%sql
DROP CATALOG IF EXISTS football_dev CASCADE;


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS football_dev
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/football'
COMMENT 'Catalogo football_dev: dominio futbol + ambiente dev';


In [0]:
%sql
DROP SCHEMA IF EXISTS football_dev.raw;
DROP SCHEMA IF EXISTS football_dev.bronze;
DROP SCHEMA IF EXISTS football_dev.silver;
DROP SCHEMA IF EXISTS football_dev.gold;


In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/football", True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/football", True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/football", True)


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS football_dev.raw;
CREATE SCHEMA IF NOT EXISTS football_dev.bronze;
CREATE SCHEMA IF NOT EXISTS football_dev.silver;
CREATE SCHEMA IF NOT EXISTS football_dev.gold;


### Tablas Bronze


In [0]:
%sql
CREATE TABLE IF NOT EXISTS football_dev.bronze.leagues (
  league_id integer,
  league_ref string,
  name string,
  country string,
  country_code string,
  confederation string,
  tier integer,
  founded_year integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/football/leagues" 


In [0]:
%sql
CREATE TABLE IF NOT EXISTS football_dev.bronze.matches (
  match_id integer,
  season_year integer,
  round integer,
  match_date date,
  league_id integer,
  home_club string,
  away_club string,
  home_goals integer,
  away_goals integer,
  ht_home_goals integer,
  ht_away_goals integer,
  ingestion_date timestamp
)
USING DELTA
PARTITIONED BY (season_year)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/football/matches" 


In [0]:
%sql
CREATE TABLE IF NOT EXISTS football_dev.bronze.clubs (
  club_id integer,
  club_ref string,
  name string,
  country string,
  city string,
  founded_year integer,
  league_id integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/football/clubs" 


### Tablas Silver


In [0]:
%sql
CREATE TABLE IF NOT EXISTS football_dev.silver.matches_transformed (
  match_id integer,
  season_year integer,
  round integer,
  match_date date,
  league_id integer,
  league_name string,
  country string,
  home_club string,
  away_club string,
  home_club_ref string,
  away_club_ref string,
  home_city string,
  away_city string,
  home_goals integer,
  away_goals integer,
  total_goals integer,
  goal_diff integer,
  result_type string,
  goal_diff_category string,
  match_intensity string,
  is_classic string,
  season_age integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/football/matches_transformed" 


### Tablas Golden


In [0]:
%sql
CREATE TABLE IF NOT EXISTS football_dev.gold.season_stats (
  season_year integer,
  country string,
  league_name string,
  conteo long,
  total_goals long,
  max_goals integer,
  min_goals integer,
  classic_count long,
  home_win_count long
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/football/season_stats" 
